In [ ]:
"""
load_data.py
-----------------------------------------
Hilfsmodul zum Laden großer TED-CSV-Datensätze
aus der offiziellen EU-Open-Data-Plattform.

Datenquelle:
https://data.europa.eu/data/datasets/ted-csv/

Hinweise:
- TED-Dateien sind sehr groß (mehrere GB).
- Chunk-Loading wird empfohlen.
- Dieses Skript lädt die Daten effizient und vereinheitlicht grundlegende Datentypen.
"""

import os
import pandas as pd
from typing import Optional, List


def load_ted_csv(
    file_path: str,
    chunksize: int = 100_000,
    usecols: Optional[List[str]] = None,
    dtype: Optional[dict] = None,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Lädt große TED-CSV-Dateien effizient mit Chunk-Loading.

    Parameter
    ---------
    file_path : str
        Pfad zur lokalen CSV-Datei (nicht im Repository gespeichert).
    chunksize : int
        Anzahl der Zeilen pro Chunk (Standard: 100.000).
    usecols : list[str], optional
        Liste der zu ladenden Spalten (falls nur Teilmenge benötigt).
    dtype : dict, optional
        Datentypen für bestimmte Spalten.
    verbose : bool
        Fortschrittsausgabe aktivieren/deaktivieren.

    Rückgabe
    --------
    pd.DataFrame
        Vollständig geladener TED-Datensatz.
    """

    # Pfad prüfen
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Datei nicht gefunden: {file_path}\n"
            "Bitte lade die Daten direkt von https://data.europa.eu/data/datasets/ted-csv/"
        )

    if verbose:
        print(f"📥 Lade TED-Daten aus: {file_path}")
        print(f"📦 Chunkgröße: {chunksize:,} Zeilen")

    # Liste für Chunks
    chunks = []

    # CSV iterativ laden
    for i, chunk in enumerate(
        pd.read_csv(
            file_path,
            sep=",",
            encoding="utf-8",
            low_memory=False,
            chunksize=chunksize,
            usecols=usecols,
            dtype=dtype
        )
    ):
        chunks.append(chunk)

        if verbose:
            print(f"🔹 Chunk {i+1} geladen ({len(chunk):,} Zeilen)")

    # Chunks zusammenführen
    df = pd.concat(chunks, ignore_index=True)

    if verbose:
        print(f"✅ Laden abgeschlossen: {len(df):,} Gesamtzeilen")

    return df


def load_multiple_years(
    folder_path: str,
    years: List[int],
    file_prefix: str = "ted",
    file_suffix: str = ".csv",
    **kwargs
) -> pd.DataFrame:
    """
    Lädt mehrere Jahresdateien (z. B. 2014–2016) und kombiniert sie.

    Parameter
    ---------
    folder_path : str
        Ordner, in dem die CSV-Dateien liegen.
    years : list[int]
        Liste der Jahre, die geladen werden sollen.
    file_prefix : str
        Dateipräfix (Standard: "ted").
    file_suffix : str
        Dateiendung (Standard: ".csv").
    kwargs :
        Zusätzliche Parameter für load_ted_csv().

    Rückgabe
    --------
    pd.DataFrame
        Kombinierter Datensatz aller geladenen Jahre.
    """

    dfs = []

    for year in years:
        file_path = os.path.join(folder_path, f"{file_prefix}_{year}{file_suffix}")

        print(f"\n📅 Lade Jahr {year} …")
        df_year = load_ted_csv(file_path, **kwargs)
        df_year["YEAR"] = year  # Falls YEAR nicht im Datensatz enthalten ist

        dfs.append(df_year)

    df_all = pd.concat(dfs, ignore_index=True)

    print(f"\n📊 Gesamtdatensatz erstellt: {len(df_all):,} Zeilen aus {len(years)} Jahren")

    return df_all


if __name__ == "__main__":
    """
    Beispielaufruf (lokal ausführen):

    python load_data.py

    Hinweis:
    Die Daten müssen vorher manuell von der EU-Open-Data-Plattform heruntergeladen werden.
    """

    # Beispiel: Laden der Jahre 2014–2016
    folder = "/path/to/local/ted_data/"
    years = [2014, 2015, 2016]

    df = load_multiple_years(
        folder_path=folder,
        years=years,
        chunksize=200_000,
        verbose=True
    )

    print(df.head())
